In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It installs the plumed Python package and generates
# synthetic COLVAR/KERNELS files used in the analysis sections, so that every
# code cell runs without requiring a local PLUMED or GROMACS installation.
#
# To use actual plumed driver on a cluster or local machine:
#   conda install -c conda-forge plumed
# or, for GROMACS-patched PLUMED:
#   follow https://www.plumed.org/doc-v2.9/user-doc/html/installation.html
# ──────────────────────────────────────────────────────────────────────────────

import subprocess, sys, os, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Install plumed Python bindings (provides the Python API; the standalone
# plumed binary must be installed separately for driver use)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plumed'], check=False)

# ── Generate synthetic data used in later sections ────────────────────────────
# Alanine dipeptide phi/psi angles drawn from a two-state distribution
# mimicking the alpha-helix and beta-sheet basins.
np.random.seed(42)
n_frames = 5000
time = np.arange(n_frames) * 0.2   # 0.2 ps stride

# Assign frames to basins: 60% alpha-helix, 40% beta-sheet
basin = np.random.choice(['alpha', 'beta'], size=n_frames, p=[0.6, 0.4])
phi = np.where(basin == 'alpha',
               np.random.normal(-1.05, 0.25, n_frames),   # alpha: ~-60 deg
               np.random.normal(-2.36, 0.30, n_frames))   # beta:  ~-135 deg
psi = np.where(basin == 'alpha',
               np.random.normal(-0.70, 0.25, n_frames),
               np.random.normal( 2.44, 0.30, n_frames))
# Wrap to [-pi, pi]
phi = (phi + np.pi) % (2*np.pi) - np.pi
psi = (psi + np.pi) % (2*np.pi) - np.pi

# ── COLVAR file from plumed driver ────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
with open('data/COLVAR_driver', 'w') as f:
    f.write('#! FIELDS time phi psi\n')
    f.write('#! SET min_phi -pi\n')
    f.write('#! SET max_phi  pi\n')
    f.write('#! SET min_psi -pi\n')
    f.write('#! SET max_psi  pi\n')
    for i in range(n_frames):
        f.write(f'{time[i]:.1f}  {phi[i]:.4f}  {psi[i]:.4f}\n')

# ── OPES_METAD COLVAR file ─────────────────────────────────────────────────────
# Includes bias and reweighting columns as PLUMED would write them
n_opes = 10000
time_opes = np.arange(n_opes) * 0.2
basin_opes = np.random.choice(['alpha','beta'], size=n_opes, p=[0.5,0.5])
phi_opes = np.where(basin_opes=='alpha',
                    np.random.normal(-1.05,0.22,n_opes),
                    np.random.normal(-2.36,0.25,n_opes))
psi_opes = np.where(basin_opes=='alpha',
                    np.random.normal(-0.70,0.22,n_opes),
                    np.random.normal( 2.44,0.25,n_opes))
phi_opes = (phi_opes+np.pi)%(2*np.pi)-np.pi
psi_opes = (psi_opes+np.pi)%(2*np.pi)-np.pi

# Toy bias: parabolic well that flattens over time (mimics OPES convergence)
# At convergence the bias approximately equals the free energy
kT = 2.479   # kJ/mol at 298 K
# True FES (two Gaussian basins)
def true_fes(ph, ps):
    g1 = -8.0*np.exp(-((ph+1.05)**2/0.12 + (ps+0.70)**2/0.12))
    g2 = -4.0*np.exp(-((ph+2.36)**2/0.15 + (ps-2.44)**2/0.15))
    return -(g1+g2)

fes_vals = true_fes(phi_opes, psi_opes)
# Scale convergence: bias builds up over time
ramp = np.clip(time_opes / (time_opes[-1]*0.6), 0, 1)
bias = -fes_vals * ramp + np.random.normal(0, 0.2, n_opes)
logweight = bias / kT
# OPES also writes opes.bias and opes.rct (reweighting constant)
rct = kT * np.log(np.mean(np.exp(logweight - logweight.max()))) + logweight.max()*kT

with open('data/COLVAR_opes_metad', 'w') as f:
    f.write('#! FIELDS time phi psi opes.bias opes.rct opes.zed opes.neff opes.nker\n')
    for i in range(n_opes):
        neff = max(1, int(50*(1-np.exp(-time_opes[i]/500))))
        nker = max(1, int(time_opes[i]/10))
        f.write(f'{time_opes[i]:.1f}  {phi_opes[i]:.4f}  {psi_opes[i]:.4f}  '
                f'{bias[i]:.4f}  {rct:.4f}  1.0  {neff}  {nker}\n')

# ── OPES_EXPANDED COLVAR file ─────────────────────────────────────────────────
# Multithermal simulation: temperatures 300-600 K
# Each frame gets a set of expansion weights for each target temperature
n_exp = 8000
time_exp = np.arange(n_exp) * 0.2
temps = [300, 350, 400, 450, 500, 600]
kB = 8.314e-3  # kJ/(mol*K)
beta_sim = 1/(kB*300)
# Simulated energies from a distribution
ene_exp = np.random.normal(-12000, 150, n_exp)
# Expansion weights (log): w_T = (beta_sim - beta_T)*U
expansion_weights = np.column_stack([
    (beta_sim - 1/(kB*T))*ene_exp for T in temps
])
# Toy bias from OPES_EXPANDED
bias_exp = kT * np.log(np.mean(np.exp(expansion_weights), axis=1) + 1e-10)
logw_exp = bias_exp / kT

with open('data/COLVAR_opes_expanded', 'w') as f:
    fields = 'time ene ' + ' '.join([f'ecv.kt_{T}' for T in temps]) + ' opes.bias opes.rct\n'
    f.write('#! FIELDS ' + fields)
    for i in range(n_exp):
        ew_str = '  '.join([f'{expansion_weights[i,j]:.4f}' for j in range(len(temps))])
        f.write(f'{time_exp[i]:.1f}  {ene_exp[i]:.2f}  {ew_str}  {bias_exp[i]:.4f}  0.0\n')

print('Setup complete. Synthetic data written to data/')
print(f'  data/COLVAR_driver          ({n_frames} frames)')
print(f'  data/COLVAR_opes_metad      ({n_opes} frames)')
print(f'  data/COLVAR_opes_expanded   ({n_exp} frames)')

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jolayfield/chem-lab-tutorials/blob/main/Track3_PLUMED/plumed_tutorial.ipynb)

# PLUMED: Collective Variables, Analysis, and OPES Enhanced Sampling

This notebook introduces PLUMED, a library for computing collective variables (CVs) and running enhanced sampling simulations. It assumes familiarity with MD simulations (forces, trajectories, GROMACS input files) but does not assume prior PLUMED experience.

| Section | Topic |
|---------|-------|
| 1 | PLUMED overview and input file anatomy |
| 2 | `plumed driver` — extracting CVs from existing trajectories |
| 3 | Collective variable reference |
| 4 | OPES_METAD — on-the-fly probability enhanced sampling |
| 5 | OPES_EXPANDED — expanded ensemble simulations |
| 6 | Post-processing and free energy surfaces |

**Running on an HPC:** All PLUMED input files shown here are production-ready. The analysis cells use synthetic COLVAR data generated in the setup cell, so you can work through the notebook without a PLUMED installation. When you run real simulations, replace the `data/COLVAR_*` paths with your actual output files.

**PLUMED version:** Examples target PLUMED 2.9. OPES became stable in 2.7; any version ≥ 2.7 will work for the OPES sections.

---
## Section 1 — PLUMED Overview and Input File Anatomy

PLUMED is a plugin that attaches to an MD engine (GROMACS, NAMD, LAMMPS, etc.) and extends it with two capabilities:

1. **Analysis** — compute any collective variable on-the-fly or in post-processing
2. **Biasing** — add forces to the simulation to enhance sampling along chosen CVs

PLUMED communicates with GROMACS through a patch applied at compile time. Once patched, you pass a `plumed.dat` input file to `gmx mdrun` using the `-plumed` flag and PLUMED acts on every MD step.

### 1.1 The PLUMED input file

A PLUMED input file is a plain text file, conventionally named `plumed.dat`. Each non-blank, non-comment line is one **action**. Actions are executed in order on every step.

The general syntax for an action is:

```
LABEL: ACTION KEYWORD1=value1 KEYWORD2=value2 ...
```

or equivalently using a multi-line block:

```
LABEL: ACTION ...
  KEYWORD1=value1
  KEYWORD2=value2
... LABEL
```

The label names the output of that action. Other actions can reference it by name. Comments start with `#`.

**Units:** PLUMED uses kJ/mol for energy, nm for length, ps for time, and radians for angles — consistent with GROMACS defaults. Distances entered as `ATOMS=1,5` refer to GROMACS atom indices (1-based).

In [ ]:
# Annotated example of a minimal plumed.dat file
# (This cell just prints it — no PLUMED required)

example_plumed_dat = """
# ── plumed.dat ─────────────────────────────────────────────────────────────

# 1. Define collective variables
#    TORSION computes a dihedral angle from four atom indices.
#    Indices match the atom order in your .gro / .tpr file (1-based).
phi: TORSION ATOMS=5,7,9,15          # backbone phi of alanine dipeptide
psi: TORSION ATOMS=7,9,15,17         # backbone psi of alanine dipeptide

# 2. Bias (optional — omit for pure analysis)
#    Apply a flat-bottom restraint to keep phi near -1.0 rad
restraint: RESTRAINT ARG=phi AT=-1.0 KAPPA=100.0

# 3. Write output
#    PRINT writes the listed quantities to a file every STRIDE steps.
PRINT ARG=phi,psi,restraint.bias FILE=COLVAR STRIDE=100
# ───────────────────────────────────────────────────────────────────────────
"""

print(example_plumed_dat)

print("Key points:")
print("  - Labels (phi, psi, restraint) let you reference a CV's value elsewhere")
print("  - ARG= accepts comma-separated labels")
print("  - STRIDE= controls output frequency (in MD steps, not time)")
print("  - FILE= sets the output filename; COLVAR is conventional for CV trajectories")

---
## Section 2 — `plumed driver`: Extracting CVs from Existing Trajectories

`plumed driver` is a standalone command that reads a pre-existing trajectory and recomputes any action in a `plumed.dat` file — without running a new simulation. This is useful when you:

- forgot to compute a CV during a simulation and want to add it in post-processing
- want to compute an expensive CV (e.g., RMSD, path CV) only on a saved trajectory
- need to reweight an OPES simulation (covered in Section 6)

### 2.1 Basic driver syntax

```bash
plumed driver \
  --plumed plumed_driver.dat \
  --mf_xtc trajectory.xtc \
  --timestep 0.002 \
  --trajectory-stride 50
```

| Flag | Meaning |
|------|---------|
| `--plumed` | Path to your PLUMED input file |
| `--mf_xtc` | Trajectory file (xtc, trr, or dcd format) |
| `--timestep` | MD timestep in ps (used to compute `time` column) |
| `--trajectory-stride` | Steps between saved frames (matches `nstxout` in your mdp) |

PLUMED also needs the system topology to know atom masses and connectivity. Provide it with `--pdb reference.pdb` or by placing a `MOLINFO STRUCTURE=reference.pdb` line in your `plumed.dat`.

### 2.2 Example: computing phi/psi angles

The `plumed.dat` below computes the backbone dihedral angles of alanine dipeptide from a GROMACS trajectory. Atom indices correspond to an ACE-ALA-NME system — adjust them for your own topology.

In [ ]:
# Write a plumed driver input file and the bash command to run it.
# On a machine with plumed installed, you would run the bash command in a terminal.

driver_dat = textwrap.dedent("""
    # plumed_driver.dat — post-process a GROMACS trajectory
    # Computes phi/psi Ramachandran angles for alanine dipeptide

    # Tell PLUMED about the molecule so it can apply PBC correctly
    MOLINFO STRUCTURE=ala_dipeptide.pdb

    # Backbone torsions  (atom names: C=carbonyl C, N=amide N, CA=alpha C)
    # For ala dipeptide:  phi = C1-N2-CA2-C2,  psi = N2-CA2-C2-N3
    phi: TORSION ATOMS=@phi-2     # shorthand using residue-based notation
    psi: TORSION ATOMS=@psi-2     # residue 2 is the alanine

    # Write to file every frame (STRIDE=1 means every stored frame)
    PRINT ARG=phi,psi FILE=COLVAR STRIDE=1
""").strip()

print("=== plumed_driver.dat ===")
print(driver_dat)
print()

print("=== bash command ===")
print("plumed driver \\")
print("  --plumed plumed_driver.dat \\")
print("  --mf_xtc md.xtc \\")
print("  --timestep 0.002 \\")
print("  --trajectory-stride 50")
print()
print("Notes:")
print("  @phi-2 / @psi-2 are PLUMED shorthand: auto-selects atoms for phi/psi")
print("  of residue 2 based on MOLINFO. Requires a correctly formatted PDB.")
print("  Equivalent explicit form: TORSION ATOMS=5,7,9,15")

### 2.3 COLVAR file format

PLUMED writes a `COLVAR` file with a header row that lists every column name. The first column is always `time` in ps. Read it with `pandas.read_csv` using `comment='#'` and `sep=r'\s+'`.

The synthetic file generated in the setup cell mimics the real format exactly.

In [ ]:
def read_colvar(path):
    """Read a PLUMED COLVAR file into a pandas DataFrame.
    
    Handles the #! FIELDS header line to extract column names,
    then reads numeric data while skipping remaining comment lines.
    """
    columns = None
    with open(path) as f:
        for line in f:
            if line.startswith('#! FIELDS'):
                columns = line.split()[2:]  # skip '#!' and 'FIELDS'
                break
    df = pd.read_csv(path, comment='#', sep=r'\s+', header=None, names=columns)
    return df

colvar = read_colvar('data/COLVAR_driver')
print(f'Loaded {len(colvar)} frames')
print(f'Columns: {list(colvar.columns)}')
print()
print(colvar.head())

In [ ]:
# Time series of phi and psi angles
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(colvar['time'], np.degrees(colvar['phi']), lw=0.5, color='steelblue', alpha=0.7)
axes[0].set_ylabel('phi (degrees)')
axes[0].axhline(-60,  color='steelblue', lw=1, ls='--', label='alpha-helix')
axes[0].axhline(-135, color='darkorange', lw=1, ls='--', label='beta-sheet')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-200, 200)

axes[1].plot(colvar['time'], np.degrees(colvar['psi']), lw=0.5, color='coral', alpha=0.7)
axes[1].set_ylabel('psi (degrees)')
axes[1].set_xlabel('time (ps)')
axes[1].set_ylim(-200, 200)

fig.suptitle('Backbone dihedral angles from plumed driver', fontsize=12)
plt.tight_layout()
plt.show()

print('Fraction of time in alpha-helix basin (phi ~ -60 deg):', 
      np.mean(np.degrees(colvar['phi']) > -100))

In [ ]:
# Ramachandran plot from the unbiased trajectory
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(np.degrees(colvar['phi']), np.degrees(colvar['psi']),
           s=1, alpha=0.3, color='steelblue')

# Annotate known basins
ax.annotate('alpha-helix\n(-60, -45)', xy=(-60, -45), fontsize=9,
            ha='center', color='steelblue',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
ax.annotate('beta-sheet\n(-135, +140)', xy=(-135, 140), fontsize=9,
            ha='center', color='darkorange',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xlabel('phi (degrees)')
ax.set_ylabel('psi (degrees)')
ax.set_title('Ramachandran plot — unbiased trajectory')
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
plt.tight_layout()
plt.show()

print("Note: the unbiased simulation is trapped — transitions between basins are rare.")
print("OPES will flatten this landscape so both basins are sampled equally.")

---
## Section 3 — Collective Variable Reference

PLUMED provides ~100 built-in CV types. The table below covers the most common ones for biomolecular simulations.

| CV | Keyword | Key arguments | Units |
|----|---------|---------------|-------|
| Distance | `DISTANCE` | `ATOMS=i,j` | nm |
| Angle | `ANGLE` | `ATOMS=i,j,k` | rad |
| Torsion (dihedral) | `TORSION` | `ATOMS=i,j,k,l` | rad |
| RMSD | `RMSD` | `REFERENCE=ref.pdb TYPE=OPTIMAL` | nm |
| Coordination number | `COORDINATION` | `GROUPA=... GROUPB=... R_0=` | dimensionless |
| Radius of gyration | `GYRATION` | `ATOMS=... TYPE=RADIUS` | nm |
| Secondary structure | `ALPHARMSD`, `PARABETARMSD` | `RESIDUES=...` | dimensionless |
| Custom function | `CUSTOM` | `ARG=a,b FUNC=a*sin(b) VAR=a,b` | user-defined |
| Linear combination | `COMBINE` | `ARG=... COEFFICIENTS=...` | user-defined |

### Atom selection

Atoms are specified by their 1-based index in the GROMACS topology. You can also define groups:

```
backbone: GROUP ATOMS=5-9,14-18,22-26    # explicit range
ca_atoms: GROUP ATOMS=@CA-1:50           # MOLINFO shorthand: all CA atoms in residues 1-50
```

### Combining CVs

You can build compound CVs from simpler ones:

In [ ]:
# Examples of common CV definitions — print for reference

cv_examples = {
    'Distance': 'dist: DISTANCE ATOMS=1,10',
    'Angle': 'ang: ANGLE ATOMS=1,5,10',
    'Dihedral': 'phi: TORSION ATOMS=5,7,9,15',
    'RMSD (Calpha)': textwrap.dedent("""
        rmsd_ca: RMSD
          REFERENCE=reference.pdb
          TYPE=OPTIMAL          # least-squares fit before measuring distance
          ATOMS=@CA             # only Calpha atoms (requires MOLINFO)
    """).strip(),
    'Coordination number': textwrap.dedent("""
        # Counts atoms in GROUPB within R_0 nm of atoms in GROUPA
        # Uses a smooth switching function, not a hard cutoff
        coord: COORDINATION
          GROUPA=1-10           # central atoms
          GROUPB=11-100         # surrounding atoms
          R_0=0.35              # switching distance in nm
          NN=6 MM=12            # switching function exponents
    """).strip(),
    'Custom (combine phi+psi)': textwrap.dedent("""
        phi: TORSION ATOMS=5,7,9,15
        psi: TORSION ATOMS=7,9,15,17
        # Helical content: both angles near -1 rad when in helix
        helix: CUSTOM ARG=phi,psi FUNC=cos(x+1.05)*cos(y+0.70) VAR=x,y PERIODIC=NO
    """).strip(),
    'Gyration radius': 'rg: GYRATION TYPE=RADIUS ATOMS=@protein',
}

for name, code in cv_examples.items():
    print(f'--- {name} ---')
    print(code)
    print()

---
## Section 4 — OPES_METAD: On-the-Fly Probability Enhanced Sampling

### 4.1 Background

Standard metadynamics adds Gaussian bias potentials to a CV space as the simulation proceeds, pushing the system out of already-visited configurations. OPES_METAD (On-the-fly Probability Enhanced Sampling, metadynamics variant) is a newer algorithm by Invernizzi and Parrinello that improves on standard metadynamics in several ways:

- **Target distribution:** OPES directly targets a well-tempered distribution. Instead of accumulating unconstrained Gaussians, it estimates the current probability distribution and adds a bias to flatten it toward a target (typically the well-tempered distribution with effective temperature `gamma * T`).
- **Faster convergence:** The adaptive kernel placement and compression scheme converges faster for most systems than well-tempered metadynamics.
- **No hill-height decay:** No separate `BIASFACTOR` / `TEMP` tuning needed. You specify `BARRIER`, the expected free energy barrier, and OPES adapts.
- **Better reweighting:** The bias is computed from the probability estimate, making unbiased free energy recovery more straightforward.

### 4.2 Key parameters

| Parameter | Meaning | Typical value |
|-----------|---------|---------------|
| `ARG` | CV(s) to bias | your CV label(s) |
| `PACE` | Steps between bias updates | 500 |
| `BARRIER` | Expected free energy barrier in kJ/mol | 10–80 |
| `SIGMA` | Initial kernel width (nm or rad) | 0.05–0.3 |
| `SIGMA_MIN` | Minimum kernel width (prevents over-compression) | SIGMA/10 |

**Choosing BARRIER:** Set it to a rough estimate of the largest barrier you want to cross. If unsure, over-estimate rather than under-estimate — a too-large BARRIER just means slower convergence, not incorrect results.

**Choosing SIGMA:** A good starting point is 1/3 to 1/5 of the characteristic width of a basin. Run a short unbiased simulation and look at the CV distribution to estimate this.

### 4.3 OPES_METAD input file

In [ ]:
opes_metad_dat = textwrap.dedent("""
    # plumed_opes_metad.dat
    # OPES_METAD on alanine dipeptide phi/psi — GROMACS with PLUMED 2.9

    MOLINFO STRUCTURE=ala_dipeptide.pdb

    # --- Collective variables ---
    phi: TORSION ATOMS=@phi-2
    psi: TORSION ATOMS=@psi-2

    # --- OPES_METAD bias ---
    opes: OPES_METAD ...
      ARG=phi,psi             # bias in 2D (phi, psi) space
      PACE=500                # update bias every 500 steps (= 1 ps at dt=0.002 ps)
      BARRIER=40              # expected barrier in kJ/mol
      SIGMA=0.15,0.15         # initial kernel widths in radians
      SIGMA_MIN=0.01,0.01     # minimum kernel widths after compression
      FILE=KERNELS            # KERNELS file stores the Gaussian basis set
      STATE_RFILE=OPES_STATE  # restart file (read previous state on restart)
      STATE_WFILE=OPES_STATE  # overwritten periodically with current state
      STORE_STATES            # keep old kernel files for convergence analysis
    ... OPES_METAD

    # --- Output ---
    # opes.bias: current bias potential value at each step
    # opes.rct:  current reweighting factor c(t)
    # opes.neff: effective sample size
    # opes.nker: number of kernels deposited
    PRINT ARG=phi,psi,opes.bias,opes.rct,opes.neff,opes.nker \\
          FILE=COLVAR STRIDE=100
""").strip()

print(opes_metad_dat)

# Save to file for reference
with open('data/plumed_opes_metad.dat', 'w') as f:
    f.write(opes_metad_dat + '\n')
print("\nSaved to data/plumed_opes_metad.dat")

### 4.4 Running with GROMACS

After patching GROMACS with PLUMED (done once at installation), the only change to your standard GROMACS workflow is the `-plumed` flag:

```bash
# Standard GROMACS steps (no change needed)
gmx grompp -f md.mdp -c start.gro -p topol.top -o md.tpr

# Add -plumed to mdrun
gmx mdrun -v -deffnm md -plumed plumed_opes_metad.dat
```

PLUMED will create:

- `COLVAR` — CV values and bias output at each `STRIDE`
- `KERNELS` — the Gaussian basis set that defines the current bias
- `OPES_STATE` — restart file for continuing a run

**mdp settings for OPES:** No special settings are needed. Use your normal NVT or NPT ensemble. For enhanced sampling runs, longer simulations (100 ns+) are typical, though OPES converges faster than standard metadynamics.

**Restarting:** To continue an interrupted OPES run, use `gmx mdrun` with `-cpi md.cpt` as normal and PLUMED will reload `OPES_STATE` automatically.

In [ ]:
# Load the synthetic OPES_METAD COLVAR file
opes_colvar = read_colvar('data/COLVAR_opes_metad')
print(f'Loaded {len(opes_colvar)} frames')
print(f'Columns: {list(opes_colvar.columns)}')
print()
print(opes_colvar.head())

In [ ]:
# Convergence diagnostics for OPES_METAD
# opes.neff (effective sample size) should grow over time
# opes.rct (reweighting constant c(t)) should plateau when bias has converged

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: neff growth
axes[0].plot(opes_colvar['time'], opes_colvar['opes.neff'], color='steelblue', lw=1)
axes[0].set_xlabel('time (ps)')
axes[0].set_ylabel('Neff (effective sample size)')
axes[0].set_title('OPES convergence: effective sample size')
axes[0].set_yscale('log')

# Panel 2: c(t) plateau indicates the FES has been mapped
axes[1].plot(opes_colvar['time'], opes_colvar['opes.rct'], color='coral', lw=1)
axes[1].set_xlabel('time (ps)')
axes[1].set_ylabel('rct (kJ/mol)')
axes[1].set_title('OPES convergence: reweighting constant c(t)')

plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Neff growing toward N_total means the bias has explored and flattened the landscape.")
print("  c(t) plateauing means the free energy surface is no longer changing significantly.")
print("  Both should level off before you consider the simulation converged.")

In [ ]:
# Compute and plot the free energy surface from OPES_METAD
# FES = -kT * ln(P_biased) + bias   
# Equivalently, use histogram reweighting with the logweight

kT = 2.479  # kJ/mol at 298 K

# Use only the last 50% of data (discard early non-converged region)
half = len(opes_colvar) // 2
df = opes_colvar.iloc[half:].copy()

# Reweighting: each frame gets weight exp(V_bias / kT)
# rct subtracts the normalization constant to keep weights near 1
logw = (df['opes.bias'] - df['opes.rct']) / kT
logw -= logw.max()  # numerical stability
weights = np.exp(logw)

# 2D histogram with weights
n_bins = 50
bins_phi = np.linspace(-np.pi, np.pi, n_bins+1)
bins_psi = np.linspace(-np.pi, np.pi, n_bins+1)

hist, xe, ye = np.histogram2d(
    df['phi'], df['psi'],
    bins=[bins_phi, bins_psi],
    weights=weights,
    density=False
)

# Convert probability to free energy: F = -kT ln(P)
hist = hist / hist.sum()
hist = np.where(hist > 0, hist, np.nan)
fes = -kT * np.log(hist)
fes -= np.nanmin(fes)   # set minimum to 0

phi_centers = (xe[:-1] + xe[1:]) / 2
psi_centers = (ye[:-1] + ye[1:]) / 2

fig, ax = plt.subplots(figsize=(7, 6))
cmap = plt.cm.viridis_r
cf = ax.contourf(
    np.degrees(phi_centers),
    np.degrees(psi_centers),
    fes.T,
    levels=np.linspace(0, 20, 21),
    cmap=cmap,
    extend='max'
)
ax.contour(
    np.degrees(phi_centers),
    np.degrees(psi_centers),
    fes.T,
    levels=np.arange(0, 22, 4),
    colors='white', linewidths=0.5, alpha=0.6
)
plt.colorbar(cf, ax=ax, label='FES (kJ/mol)')
ax.set_xlabel('phi (degrees)')
ax.set_ylabel('psi (degrees)')
ax.set_title('Free energy surface from OPES_METAD reweighting')
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
plt.tight_layout()
plt.show()

In [ ]:
# 1D free energy profile along phi (marginal FES)
bins_phi_1d = np.linspace(-np.pi, np.pi, 60)
hist_1d, edges = np.histogram(df['phi'], bins=bins_phi_1d, weights=weights, density=False)
hist_1d = hist_1d / hist_1d.sum()
hist_1d = np.where(hist_1d > 0, hist_1d, np.nan)
fes_1d = -kT * np.log(hist_1d)
fes_1d -= np.nanmin(fes_1d)
phi_mid = np.degrees((edges[:-1] + edges[1:]) / 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(phi_mid, fes_1d, color='steelblue', lw=2)
ax.set_xlabel('phi (degrees)')
ax.set_ylabel('F(phi) — kJ/mol')
ax.set_title('Marginal free energy along phi from OPES_METAD')
ax.set_xlim(-180, 180)
ax.axvline(-60,  color='gray', ls='--', lw=1, label='alpha-helix basin')
ax.axvline(-135, color='gray', ls=':', lw=1, label='beta-sheet basin')
ax.legend()
plt.tight_layout()
plt.show()

# Compute the free energy difference between basins
# Alpha-helix basin: phi in [-100, -20] degrees
# Beta-sheet basin:  phi in [-180, -100] degrees
mask_alpha = (phi_mid >= -100) & (phi_mid <= -20)
mask_beta  = (phi_mid >= -180) & (phi_mid <= -100)
dF = np.nanmin(fes_1d[mask_beta]) - np.nanmin(fes_1d[mask_alpha])
print(f'DeltaF (beta - alpha) from marginal FES: {dF:.2f} kJ/mol')

---
## Section 5 — OPES_EXPANDED: Expanded Ensemble Simulations

### 5.1 Background

OPES_EXPANDED implements the expanded ensemble idea within the OPES framework. Instead of biasing a structural CV (like OPES_METAD), it biases a **thermodynamic parameter** — temperature, Hamiltonian, or a physical CV — to sample multiple thermodynamic states in a single simulation.

The key concept is **expanded collective variables (ECVs)**, which map thermodynamic states onto an effective CV space. OPES_EXPANDED builds a bias over this ECV space that ensures all target states are sampled.

### 5.2 When to use OPES_EXPANDED instead of OPES_METAD

| Situation | Recommended method |
|-----------|--------------------|
| Free energy along a structural CV (distance, dihedral) | OPES_METAD |
| Conformational sampling at multiple temperatures | OPES_EXPANDED with ECV_MULTITHERMAL |
| Binding free energy, alchemical transformation | OPES_EXPANDED with ECV_ALCHEMICAL |
| Combining temperature and Hamiltonian expansion | OPES_EXPANDED with multiple ECVs |
| Umbrella sampling across many lambda windows in one run | OPES_EXPANDED with ECV_UMBRELLAS_CS |

### 5.3 ECV types

| ECV keyword | Expands over | Required arguments |
|-------------|--------------|-------------------|
| `ECV_MULTITHERMAL` | Temperature range | `ARG=ene TEMP_MIN TEMP_MAX` |
| `ECV_MULTIBARIC` | Pressure range | `ARG=ene,vol PRES_MIN PRES_MAX` |
| `ECV_MULTITHERMAL_MULTIBARIC` | Temp + pressure | see docs |
| `ECV_ALCHEMICAL` | Lambda parameter | `ARG=dVdl LAMBDA_STEPS` |
| `ECV_UMBRELLAS_CS` | Umbrella window grid | `ARG=cv CENTER_SIGMA=...` |
| `ECV_LINEXP` | Linear expansion (custom) | `ARG=obs COEFFICIENTS=...` |

### 5.4 OPES_EXPANDED input — multithermal example

The multithermal variant samples all temperatures between `TEMP_MIN` and `TEMP_MAX` in a single trajectory. At the end, you can extract the properties at each temperature by reweighting.

In [ ]:
opes_expanded_multithermal = textwrap.dedent("""
    # plumed_opes_expanded_multithermal.dat
    # Multithermal OPES_EXPANDED: sample all temperatures 300–600 K in one run
    # The MD thermostat temperature should be set to TEMP_MIN (300 K here)

    # --- Potential energy (required for temperature expansion) ---
    ene: ENERGY

    # --- Expanded collective variable ---
    # ECV_MULTITHERMAL maps the energy onto an ECV for each target temperature.
    # TEMP_MIN/MAX are the temperature range to sample; the number of intermediate
    # temperatures is chosen automatically (or set with STEPS).
    ecv: ECV_MULTITHERMAL ...
      ARG=ene
      TEMP_MIN=300              # simulation temperature (K)
      TEMP_MAX=600              # highest target temperature (K)
      STEPS=20                  # number of temperature windows
    ... ECV_MULTITHERMAL

    # --- OPES_EXPANDED bias ---
    # ARG=(ecv\.*) passes all output labels from the ECV action
    opes: OPES_EXPANDED ...
      ARG=(ecv\.*)              # regex selects all ecv output labels
      PACE=500
      FILE=KERNELS
      STATE_WFILE=OPES_STATE
      STATE_RFILE=OPES_STATE
    ... OPES_EXPANDED

    # --- Output ---
    PRINT ARG=ene,opes.bias,opes.rct FILE=COLVAR STRIDE=100
""").strip()

print(opes_expanded_multithermal)
with open('data/plumed_opes_expanded_multithermal.dat', 'w') as f:
    f.write(opes_expanded_multithermal + '\n')
print("\nSaved to data/plumed_opes_expanded_multithermal.dat")

In [ ]:
# Alchemical OPES_EXPANDED — binding free energy
# Requires a GROMACS topology with a lambda group and dVdl output in the energy file

opes_expanded_alch = textwrap.dedent("""
    # plumed_opes_expanded_alch.dat
    # Alchemical OPES_EXPANDED for binding free energy
    # Requires: mdp with free-energy = yes, dhdl-print-energy = yes
    #           and a plumed-compatible energy reader

    # Read dVdl from GROMACS (dH/dlambda for the current lambda state)
    # PLUMED's ENERGY action returns the potential energy at current lambda.
    # For alchemical work, pass the full dH/dl through GROMACS_LAMBDA_ENERGY.
    dVdl: GROMACS_LAMBDA_ENERGY  # available in PLUMED 2.9+

    # ECV_ALCHEMICAL spans lambda = 0 (fully interacting) to lambda = 1 (decoupled)
    ecv: ECV_ALCHEMICAL ...
      ARG=dVdl
      LAMBDA_STEPS=20           # number of lambda windows
    ... ECV_ALCHEMICAL

    opes: OPES_EXPANDED ...
      ARG=(ecv\.*)              # all ECV outputs
      PACE=500
      FILE=KERNELS
      STATE_WFILE=OPES_STATE
      STATE_RFILE=OPES_STATE
    ... OPES_EXPANDED

    PRINT ARG=dVdl,opes.bias,opes.rct FILE=COLVAR STRIDE=100

    # NOTE: ECV_UMBRELLAS_CS is useful when you want to span a CV range
    # (e.g., a binding distance) across many umbrella windows in one run:
    #
    #   cv: DISTANCE ATOMS=1,100
    #   ecv_umb: ECV_UMBRELLAS_CS ARG=cv CENTER_SIGMA=0.1 NUMB=30 RANGE=0.2,3.0
    #   opes: OPES_EXPANDED ARG=(ecv_umb\.*) PACE=500 ...
""").strip()

print(opes_expanded_alch)

### 5.5 Running OPES_EXPANDED with GROMACS

The GROMACS command is the same as for OPES_METAD:

```bash
gmx mdrun -v -deffnm md -plumed plumed_opes_expanded_multithermal.dat
```

**For multithermal:** Set the GROMACS thermostat temperature to `TEMP_MIN`. PLUMED's bias handles the effective temperature excursions. Use a weak thermostat (e.g., `v-rescale` with `tau-t = 0.5 ps`) to avoid interference with the OPES algorithm.

**For alchemical:** Set `free-energy = yes` in your MDP and select one of your normal lambda states (e.g., `init-lambda-state = 0`). PLUMED then handles the expanded sampling across all lambda windows defined in the ECV.

In [ ]:
# Analyze the OPES_EXPANDED multithermal output
# Goal: extract the average potential energy at each target temperature

exp_colvar = read_colvar('data/COLVAR_opes_expanded')
print(f'Loaded {len(exp_colvar)} frames')
print(f'Columns: {list(exp_colvar.columns)[:5]}...')
print()
print(exp_colvar.head(3))

In [ ]:
# Reweighting OPES_EXPANDED to extract thermodynamic properties at each temperature
#
# The reweighting formula for OPES_EXPANDED:
#   w_i(T) = exp( logw_opes + (1/kT_sim - 1/kT) * U_i )
#
# where logw_opes = (V_bias - c(t)) / kT_sim  is the OPES reweighting factor
# and the exponential factor accounts for the temperature change.

kB = 8.314e-3       # kJ/(mol*K)
T_sim = 300         # simulation temperature
kT_sim = kB * T_sim

target_temps = [300, 350, 400, 450, 500, 600]

# Use last 75% of simulation (post-equilibration)
n_skip = len(exp_colvar) // 4
df_exp = exp_colvar.iloc[n_skip:].copy()

# Base OPES logweight
logw_opes = (df_exp['opes.bias'] - df_exp['opes.rct']) / kT_sim

mean_energies = []
std_energies = []

for T in target_temps:
    kT = kB * T
    # Total logweight for this temperature
    logw = logw_opes + (1/kT_sim - 1/kT) * df_exp['ene']
    logw -= logw.max()  # numerical stability
    w = np.exp(logw)
    w /= w.sum()

    # Weighted average energy
    mean_E = np.sum(w * df_exp['ene'])
    # Weighted variance
    var_E  = np.sum(w * (df_exp['ene'] - mean_E)**2)
    std_E  = np.sqrt(var_E)

    mean_energies.append(mean_E)
    std_energies.append(std_E)

# Compute effective sample size for each temperature to gauge uncertainty
neff_list = []
for T in target_temps:
    kT = kB * T
    logw = logw_opes + (1/kT_sim - 1/kT) * df_exp['ene']
    logw -= logw.max()
    w = np.exp(logw)
    neff = (w.sum())**2 / (w**2).sum()
    neff_list.append(neff)

print('Temperature reweighting results:')
print(f'{"T (K)":>8}  {"<E> (kJ/mol)":>14}  {"std(E)":>10}  {"N_eff":>8}')
print('-' * 50)
for T, E, s, n in zip(target_temps, mean_energies, std_energies, neff_list):
    print(f'{T:8.0f}  {E:14.1f}  {s:10.1f}  {n:8.0f}')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(target_temps, mean_energies, yerr=std_energies,
            fmt='o-', color='steelblue', capsize=5, lw=2)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Mean potential energy (kJ/mol)')
ax.set_title('Energy vs. temperature from OPES_EXPANDED reweighting')
plt.tight_layout()
plt.show()

---
## Section 6 — Post-Processing and Free Energy Surfaces

### 6.1 Using `plumed driver` for reweighting

After an OPES simulation, you can compute additional CVs and combine them with the saved bias for reweighting — without re-running the simulation:

```bash
# plumed_reweight.dat
#   Compute a new CV (e.g., RMSD) from the OPES trajectory
#   and write it alongside the saved bias

MOLINFO STRUCTURE=reference.pdb
rmsd: RMSD REFERENCE=reference.pdb TYPE=OPTIMAL
phi: TORSION ATOMS=@phi-2

# Read the previously saved bias from the OPES run
# METAD (or OPES_METAD) can be started from a KERNELS file
opes: OPES_METAD ...
  ARG=phi
  PACE=1e10          # never update
  BARRIER=40
  SIGMA=0.15
  KERNELS_FILE=KERNELS   # load existing kernels, do not add new ones
... OPES_METAD

PRINT ARG=rmsd,phi,opes.bias FILE=COLVAR_reweight STRIDE=1
```

```bash
plumed driver --plumed plumed_reweight.dat --mf_xtc md.xtc \
              --timestep 0.002 --trajectory-stride 50
```

Then reweight `COLVAR_reweight` with the same `exp(V_bias / kT)` formula from Section 4.

### 6.2 The `sum_hills` equivalent in OPES

Traditional metadynamics used `plumed sum_hills` to reconstruct the FES from accumulated Gaussian hills. OPES does not use this tool. Instead, use histogram reweighting directly from the COLVAR file as shown in Section 4. The OPES bias at convergence already equals `-F(CV)`, so you can also plot the instantaneous bias as an approximate FES:

```python
# Quick FES estimate from the final bias value (last frame)
# Less accurate than full reweighting but useful for a fast check
# Read the KERNELS file and evaluate
```

### 6.3 Block averaging for uncertainty estimates

In [ ]:
# Block averaging on the 1D FES to estimate statistical uncertainty
# Split the converged part of the trajectory into N_blocks equal segments
# Compute the FES from each block, then take the standard deviation.

def fes_1d_from_colvar(df, col, weights, bins):
    """Compute a 1D FES from a weighted histogram."""
    h, edges = np.histogram(df[col], bins=bins, weights=weights, density=False)
    h = h / h.sum()
    h = np.where(h > 0, h, np.nan)
    fes = -kT * np.log(h)
    fes -= np.nanmin(fes)
    return fes

kT = 2.479
bins_phi = np.linspace(-np.pi, np.pi, 50)

# Use the second half of the OPES_METAD trajectory
half = len(opes_colvar) // 2
df_conv = opes_colvar.iloc[half:].copy().reset_index(drop=True)

# Compute logweights
logw = (df_conv['opes.bias'] - df_conv['opes.rct']) / kT
logw -= logw.max()
weights_all = np.exp(logw)

# Block averaging
N_blocks = 5
block_size = len(df_conv) // N_blocks
block_fes = []

for i in range(N_blocks):
    start = i * block_size
    end   = start + block_size
    df_b  = df_conv.iloc[start:end]
    w_b   = weights_all.iloc[start:end]
    block_fes.append(fes_1d_from_colvar(df_b, 'phi', w_b, bins_phi))

block_fes = np.array(block_fes)  # shape (N_blocks, n_bins)

# Align blocks to a common reference (minimum of each block = 0)
# Already done in fes_1d_from_colvar

fes_mean = np.nanmean(block_fes, axis=0)
fes_sem  = np.nanstd(block_fes, axis=0) / np.sqrt(N_blocks)
phi_centers = np.degrees((bins_phi[:-1] + bins_phi[1:]) / 2)

fig, ax = plt.subplots(figsize=(8, 4))
for i, bf in enumerate(block_fes):
    ax.plot(phi_centers, bf, lw=1, alpha=0.4, label=f'block {i+1}')
ax.plot(phi_centers, fes_mean, lw=2.5, color='black', label='mean')
ax.fill_between(phi_centers, fes_mean - fes_sem, fes_mean + fes_sem,
                color='black', alpha=0.2, label='± 1 SEM')
ax.set_xlabel('phi (degrees)')
ax.set_ylabel('F(phi) — kJ/mol')
ax.set_title('Block-averaged 1D FES with uncertainty estimate')
ax.set_xlim(-180, 180)
ax.set_ylim(-1, 22)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('Max SEM across phi bins:', np.nanmax(fes_sem).round(2), 'kJ/mol')
print('A SEM < 1 kJ/mol for the free energy difference is generally acceptable.')

---
## Section 7 — Tips, Common Mistakes, and Troubleshooting

### 7.1 Choosing CVs for OPES

The choice of CV is the most consequential decision in any enhanced sampling simulation. A bad CV can lead to slow convergence or, worse, a converged but incorrect FES. Guiding principles:

- The CV should distinguish all metastable states you want to characterize. If two states have identical CV values, OPES cannot separate them on the FES.
- Fewer CVs are better. 1D or 2D CVs are far easier to converge than 3D+. Consider whether a single well-chosen CV captures the slow degrees of freedom.
- Run a short unbiased simulation first and plot the CV. A good CV shows the system getting trapped — the very thing OPES will fix.

### 7.2 Common OPES mistakes

**PACE too small:** Updating the bias every step gives OPES no time to observe the effect of previous updates. PACE >= 500 is almost always appropriate.

**SIGMA too large:** Large initial kernels blur the FES and slow convergence. Use a value roughly 1/5 the basin width.

**BARRIER too small:** If BARRIER is smaller than the true barrier, OPES will not explore all of CV space. Always err on the side of a larger BARRIER.

**Not discarding the equilibration phase:** The bias at early times is not well-converged. Use only the latter portion of the trajectory for FES estimates (Section 4 uses the last 50%).

**Using OPES_METAD for alchemical calculations:** Use OPES_EXPANDED with ECV_ALCHEMICAL instead — it handles the expanded statistical ensemble correctly.

### 7.3 Checking your PLUMED input file

Before running a full simulation, test your input file with `plumed driver` on a short trajectory (or even a single frame) to confirm the CV values are sensible:

```bash
plumed driver --plumed plumed_opes_metad.dat \
              --mf_gro start.gro             # single frame check
```

If atom indices are wrong, PLUMED will either crash or produce obviously incorrect CV values (e.g., a dihedral of exactly 0).

### 7.4 Useful PLUMED resources

| Resource | URL |
|----------|-----|
| PLUMED documentation | https://www.plumed.org/doc-v2.9/user-doc/html/ |
| OPES module documentation | https://www.plumed.org/doc-v2.9/user-doc/html/opes.html |
| PLUMED-NEST (example inputs) | https://www.plumed-nest.org/ |
| Invernizzi & Parrinello (OPES_METAD) | J. Phys. Chem. Lett. 2020, 11, 2731 |
| Invernizzi et al. (OPES_EXPANDED) | J. Chem. Theory Comput. 2022, 18, 3988 |